# V1002 — End-to-End Walkthrough v3

v3 修复一个更根本的问题：上一版 observable oracle 把 **boundary anchor 的中心-cell label** 当成纯 niche truth。
但本方法定义的是 local neighborhood niche，因此边界 neighborhood 本来就是混合态；把它们强行要求为纯 N3/N4 会让 CCC 因 background/opportunity 混合而产生伪可分性。

v3 将 **core/interior neighborhood oracle** 作为 simulation identifiability gate：只有 neighborhood 中至少 80% 成员属于 anchor truth niche 的 anchors 才参与 hard-pair oracle。边界 anchors 保留在模型训练与 localization 评价中，不用于“纯 view 可辨识性”判定。


## 0. 当前源码审查后先记住的 5 个事实

1. `8408` 是完整 CommuSpace atlas 大小；`build_simulation_spec()` 先用 assay-gene 约束得到实际进入 simulation 的子 atlas。当前代码中通常约为 **849 LR**，所以真正生成的 assay-level directed feature space 是 \(64\times849\)，不是直接构造 538,112 个 dense features。
2. Primary setting 有 1800 cells，5 个 niche × 100 cells，因此 background 大约占 72%。这是一个强烈的 unsupervised reconstruction imbalance。
3. `N1/N2` 被设计成 composition twin；`N3/N4` 被设计成 CCC twin。但“generator truth 相同/不同”不等于最终 neighborhood-level \(C_S/I_S\) 也相同/不同，必须先做 oracle audit。
4. `fit_balanced()` 平衡的是 reconstruction gradient magnitude，不是每个 view 对 niche identity 的信息量。
5. 源码 `evaluate()` 对 `HC` 使用 generator-level `hc_truth` 做 factor matching，而对 `HI` 使用 observed niche centroid。Notebook 会额外做 **activity-oracle alignment**，检查结果差是不是仅由 factor matching 造成。

我对上传代码做了一个很轻量的 composition-only 检查：在不生成 CCC 的情况下，N3/N4 的 observed neighborhood composition 用 logistic regression 可以达到约 **0.95–0.99 AUC**，但 composition-only NMF 的 niche sensitivity 仍然很低。也就是说，至少对这一组 hard case，问题不只是“composition 被 neighborhood 冲淡”，而是 **可分信息并不保证被 reconstruction-NMF 分配成一个 niche factor**。

In [ ]:
from pathlib import Path
import os, sys, json, math, time, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from IPython.display import display

CANDIDATES = [
    Path.cwd(),
    Path('/home/xueshuailin/CCC_Phe/V1002'),
    Path('/mnt/data/V1002_src/V1002'),
]
PROJECT_ROOT = next((p.resolve() for p in CANDIDATES if (p / 'src/phenoniche/v1002').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 V1002 项目根目录；请把 notebook 放到 V1002 根目录或修改 CANDIDATES。')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DEVICE       =', DEVICE)
print('torch        =', torch.__version__)

In [ ]:
# 主开关
PURITY = 0.50
NICHE_SIZE = 100
NOISE = 0.05
PATIENTS = 320
MODEL_SEED = 31

# observable identifiability 必须先通过；这是 outcome-blind generator gate
AUTO_REPAIR_ORACLE = True
ORACLE_SEEDS = [33700, 33701, 33702]
DATA_SEED = ORACLE_SEEDS[0]  # backward-compatible alias used by lightweight/bulk cells
MAX_REPAIR_VARIANT = 5

# 若 AUTO_REPAIR_ORACLE=False，可手动指定；0 是已知失败的 historical generator
REPAIR_VARIANT = 0

# walkthrough 默认只对冻结后的第一份 tissue 展开 ST；三 seed 汇总可在后面打开
RUN_FULL_PRIMARY = True
RUN_THREE_SEED_ST = True
RUN_ROBUSTNESS = False


## 1. 导入 V1002 当前实现

这里全部调用你上传项目里的真实代码，而不是我重新写一套“看起来类似”的实现。

In [ ]:
from phenoniche.v1002.lr_atlas import load_lr_atlas, feature_coordinates
from phenoniche.v1002.simulation_cells import (
    _layout, _cell_types, _coverage_mask, _aggregate_views, bulk_potential,
)
from phenoniche.v1002.final_simulation import (
    build_final_spec, simulate_final_spatial, simulate_final_bulk,
    true_activity, alr, TRUE_BETA, _composition_v3, _quota_types,
)
from phenoniche.v1001.neighborhoods import build_neighborhoods
from phenoniche.v1002.final_experiment import make_blocks, filtering, collinearity
from phenoniche.v1002.final_model import fit_balanced, evaluate
from phenoniche.v1002.simulation_model import infer_activity, _correlation
from phenoniche.v1002.oracle_identifiability import heldout_auc, audit_replicates, oracle_pass
from phenoniche.evaluation.patient_protocol import fit_frozen_cox, split_patients
from phenoniche.evaluation.metrics import concordance_index
from phenoniche.losses.survival import cox_breslow_loss


## 2. LR atlas：先搞清楚 8408、assay-measurable LR 和 directed CCC 的关系

完整 atlas 是候选知识库；simulation 并没有把 8408×64 全部 materialize。`build_final_spec()` 先构造一个 assay-gene vocabulary，再保留所有 component 都可测的 LR。

In [ ]:
atlas = load_lr_atlas('data/commuspace_human_lr_atlas.tsv')
spec = build_final_spec(atlas)

atlas_summary = pd.DataFrame({
    'quantity': [
        'Full LR atlas', 'Simulation assay-measurable LR', 'LR-related genes',
        'Raw directed CCC candidates', 'Assay-level directed CCC', 'True active records'
    ],
    'value': [
        len(atlas), len(spec.atlas), len(spec.genes),
        64 * len(atlas), 64 * len(spec.atlas), len(spec.active_edges)
    ]
})
display(atlas_summary)
print('spec.hi_truth shape      =', spec.hi_truth.shape)
print('spec.expression_delta    =', spec.expression_delta.shape)

### Feature identity

一个 CCC feature 的扁平 index 对应：

\[
(a\to b, LR_l),\qquad
f=((a\cdot C)+b)\cdot L+l.
\]

下面随便解码几个 active edges，确认 sender / receiver / LR 三层语义没有丢。

In [ ]:
rows=[]
for niche, feature, strength, kind in spec.active_edges[:12]:
    sender, receiver, lr_idx = feature_coordinates(int(feature), 8, len(spec.atlas))
    lr = spec.atlas.interactions[lr_idx]
    rows.append({
        'niche': niche + 1,
        'sender': sender,
        'receiver': receiver,
        'LR': f'{lr.ligand} -> {lr.receptor}',
        'strength': strength,
        'kind': kind,
    })
display(pd.DataFrame(rows))

## 3. 先不生成 CCC：做一个便宜但关键的 composition sanity check

这是我认为现在最重要的诊断之一。

如果 N3/N4 在 **observed neighborhood composition** 上已经高度可分，但 composition-only NMF 仍无法形成 N3/N4 factors，那么不能再把失败简单归因为“simulation 没把 composition signal 生成出来”。

In [ ]:
def lightweight_composition_sim(repair=0, seed=None):
    if seed is None:
        seed = ORACLE_SEEDS[0]
    coordinates, labels = _layout(NICHE_SIZE, seed)
    hc = _composition_v3(PURITY, pair_b=repair >= 2, stronger_pair_b=repair >= 4)
    cell_types = _cell_types(labels, hc, seed + 101)
    if repair > 0:
        groups = [(1, 2)] if repair == 1 else [(1, 2), (3,), (4,)]
        cell_types = _quota_types(
            cell_types, labels, coordinates,
            {1: hc[0], 3: hc[2], 4: hc[3]}, seed + 303, groups
        )
    k = 12 if repair >= 5 else 15
    neighborhoods = build_neighborhoods(coordinates, k=k, sigma=.8)
    members = neighborhoods.indices.reshape(len(labels), k)
    weights = neighborhoods.weights.reshape(len(labels), k).astype(np.float32)
    local_types = cell_types[members]
    cs = np.stack([
        ((local_types == t) * weights).sum(1) / weights.sum(1)
        for t in range(8)
    ], axis=1).astype(np.float32)
    class S: pass
    s=S(); s.coordinates=coordinates; s.labels=labels; s.cell_types=cell_types
    s.hc_truth=hc; s.neighborhoods=neighborhoods
    return s, cs

light_s, light_cs = lightweight_composition_sim(REPAIR_VARIANT)
light_ws = true_activity(light_s)

auc_12_cs = heldout_auc(light_cs, light_s.labels, light_s.coordinates, 1, 2)
auc_34_cs = heldout_auc(light_cs, light_s.labels, light_s.coordinates, 3, 4)
comp_fit_light = fit_balanced({'HC': light_cs}, seed=MODEL_SEED, device=DEVICE)
comp_report_light = evaluate(comp_fit_light, light_s, light_ws, {'HC': light_cs})

print('Observed CS AUC, N1 vs N2 =', round(auc_12_cs, 3), '(应该接近随机)')
print('Observed CS AUC, N3 vs N4 =', round(auc_34_cs, 3), '(设计目标：高)')
print('C-only NMF sensitivities =',
      [round(x['sensitivity'], 3) for x in comp_report_light['per_factor']])
print('C-only overall WS =', round(comp_report_light['overall_ws'], 3))


### 如何读上面的结果

- `CS AUC(N3,N4)` 高：说明 **模型输入本身有 composition 区分信息**。
- 但 composition-only NMF sensitivity 很低：说明 **low-rank reconstruction 并不会自动把一个可分类方向变成一个独立 niche factor**。

这是当前 V1002 很容易被忽略的一点：

\[
\text{observable separability} \not\Rightarrow \text{NMF factor identifiability}.
\]

这也是为什么后面必须同时看 oracle AUC 和 factorization recovery。

### 3A.1 Core-neighborhood oracle：把 boundary mixture 与 intrinsic view identifiability 分开

定义 core anchor：其 kNN neighborhood 中至少 `CORE_FRAC=0.80` 的成员来自同一 truth niche。

Hard-pair oracle 只在 core anchors 上判断：

- N1/N2：CS 应不可分、IS 应可分；
- N3/N4：CS 应可分、IS 应不可分。

边界 anchors 不删除，仍参与后续 NMF / Full 模型训练；这里只是不把混合 neighborhood 当作纯 niche 来审计 generator。


In [ ]:
CORE_FRAC = 0.80

def core_anchor_mask(simulation, frac=CORE_FRAC):
    members = simulation.neighborhoods.indices.reshape(len(simulation.labels), -1)
    local_same = np.mean(
        simulation.labels[members] == simulation.labels[:, None],
        axis=1
    )
    return local_same >= frac

def audit_replicates_core(simulations, seeds, core_frac=CORE_FRAC):
    if len(simulations) != 3:
        raise ValueError("Core oracle expects exactly three tissues")

    common = np.logical_and.reduce([s.final_mask.ravel() for s in simulations])
    features = np.flatnonzero(common)

    cs = [s.cs for s in simulations]
    interaction = [s.communication[:, features] for s in simulations]
    labels = [s.labels for s in simulations]
    cores = [core_anchor_mask(s, core_frac) for s in simulations]

    def select(j, positive, negative, background=False):
        if background:
            return cores[j]
        return cores[j] & ((labels[j] == positive) | (labels[j] == negative))

    def target(y, positive, background=False):
        return (y != 0).astype(np.int8) if background else (y == positive).astype(np.int8)

    def auc(view, held, positive, negative, background=False):
        train_x, train_y = [], []
        for j in range(3):
            if j == held:
                continue
            m = select(j, positive, negative, background)
            train_x.append(view[j][m])
            train_y.append(target(labels[j][m], positive, background))
        train_x = np.concatenate(train_x)
        train_y = np.concatenate(train_y)

        m = select(held, positive, negative, background)
        test_x = view[held][m]
        test_y = target(labels[held][m], positive, background)

        scale = np.sqrt(np.mean(train_x.astype(np.float64) ** 2))
        model = LogisticRegression(
            C=1.0, solver="liblinear", class_weight="balanced",
            max_iter=1000, random_state=9001
        )
        model.fit(train_x / max(scale, 1e-12), train_y)
        score = model.predict_proba(test_x / max(scale, 1e-12))[:, 1]
        return float(roc_auc_score(test_y, score))

    rows = []
    for held in range(3):
        combined = [
            np.concatenate((
                c / max(np.sqrt(np.mean(c.astype(np.float64) ** 2)), 1e-12),
                i / max(np.sqrt(np.mean(i.astype(np.float64) ** 2)), 1e-12)
            ), axis=1)
            for c, i in zip(cs, interaction)
        ]
        rows.append({
            "seed": seeds[held],
            "N1_N2": {
                "CS_AUC": auc(cs, held, 1, 2),
                "IS_AUC": auc(interaction, held, 1, 2),
            },
            "N3_N4": {
                "CS_AUC": auc(cs, held, 3, 4),
                "IS_AUC": auc(interaction, held, 3, 4),
            },
            "background_vs_niche_CS_IS_AUC": auc(combined, held, 1, 0, background=True),
            "core_counts": {
                str(k): int(np.sum(cores[held] & (labels[held] == k)))
                for k in range(6)
            }
        })

    return {
        "rows": rows,
        "common_feature_count": len(features),
        "common_feature_indices": features,
        "core_fraction_threshold": core_frac,
    }

def oracle_pass_core(rows):
    return all(
        max(r["N1_N2"]["CS_AUC"], 1-r["N1_N2"]["CS_AUC"]) <= 0.60
        and r["N1_N2"]["IS_AUC"] >= 0.90
        and r["N3_N4"]["CS_AUC"] >= 0.90
        and max(r["N3_N4"]["IS_AUC"], 1-r["N3_N4"]["IS_AUC"]) <= 0.60
        and r["background_vs_niche_CS_IS_AUC"] >= 0.90
        for r in rows
    )


## 3B. 先自动修 simulation observable，再冻结数据

这一节是 v2 最重要的变化。`repair=0` 本来就是已知失败基线，所以不能拿它直接判断 NMF。

我们对 repair 0→5 依次生成 **3 个独立 tissue**，使用 leave-one-tissue-out oracle。选择规则只看 observable：

- N1/N2：CS 不可分、IS 可分；
- N3/N4：CS 可分、IS 不可分；
- Background vs niche 可分。

一旦通过，立即冻结该 variant；后续模型不能再修改 generator。


In [ ]:
import gc

selected_variant = None
selected_simulations = None
selected_oracle = None
selected_oracle_all_anchor = None
oracle_attempts = []

if AUTO_REPAIR_ORACLE:
    supports_repair = "repair" in inspect.signature(simulate_final_spatial).parameters
    variants = range(MAX_REPAIR_VARIANT + 1) if supports_repair else (REPAIR_VARIANT,)
    if not supports_repair:
        print("Current simulation has no repair parameter; auditing the fixed generator once.")
    for variant in variants:
        print(f"\n===== observable oracle: repair={variant} =====")
        sims = [
            simulate_final_spatial(
                spec, PURITY, NICHE_SIZE, NOISE, seed,
                **({"device": DEVICE, "repair": variant} if supports_repair else {"device": DEVICE})
            )
            for seed in ORACLE_SEEDS
        ]

        # Primary gate: pure/core neighborhoods only.
        audit_core = audit_replicates_core(sims, ORACLE_SEEDS)
        passed = oracle_pass_core(audit_core["rows"])

        core_table = pd.DataFrame([
            {
                "seed": r["seed"],
                "N12_CS": r["N1_N2"]["CS_AUC"],
                "N12_CS_sym": max(r["N1_N2"]["CS_AUC"], 1-r["N1_N2"]["CS_AUC"]),
                "N12_IS": r["N1_N2"]["IS_AUC"],
                "N34_CS": r["N3_N4"]["CS_AUC"],
                "N34_IS": r["N3_N4"]["IS_AUC"],
                "N34_IS_sym": max(r["N3_N4"]["IS_AUC"], 1-r["N3_N4"]["IS_AUC"]),
                "BG_AUC": r["background_vs_niche_CS_IS_AUC"],
            }
            for r in audit_core["rows"]
        ])
        print("CORE-neighborhood oracle:")
        display(core_table)
        print("common retained CCC =", audit_core["common_feature_count"])
        print("core_oracle_pass =", passed)

        # Diagnostic only: all anchors, including boundary mixtures.
        audit_all = audit_replicates(sims, ORACLE_SEEDS)
        all_table = pd.DataFrame([
            {
                "seed": r["seed"],
                "N12_CS": r["N1_N2"]["CS_AUC"],
                "N12_IS": r["N1_N2"]["IS_AUC"],
                "N34_CS": r["N3_N4"]["CS_AUC"],
                "N34_IS": r["N3_N4"]["IS_AUC"],
                "BG_AUC": r["background_vs_niche_CS_IS_AUC"],
            }
            for r in audit_all["rows"]
        ])
        print("\nAll-anchor diagnostic (boundary included; NOT the gate):")
        display(all_table)

        oracle_attempts.append({
            "variant": variant,
            "passed": passed,
            "common_features": audit_core["common_feature_count"],
            "core_table": core_table.copy(),
            "all_anchor_table": all_table.copy(),
        })

        if passed:
            selected_variant = variant
            selected_simulations = sims
            selected_oracle = audit_core
            selected_oracle_all_anchor = audit_all
            REPAIR_VARIANT = variant
            print(f"\nFREEZE simulation at repair={variant}")
            break

        del sims, audit_core, audit_all
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if selected_variant is None:
        print("\nSTOP: repair 0..5 均未满足 CORE-neighborhood observable oracle。")
        print("此时才说明 simulation generator 仍有问题。")
else:
    print("AUTO_REPAIR_ORACLE=False；将使用手动 REPAIR_VARIANT =", REPAIR_VARIANT)


## 4. 生成完整 spatial simulation

现在才进入最耗内存的一步。数据不是直接模拟 \(C_S/I_S\)，而是：

\[
\text{coordinates}\rightarrow\text{cell type}\rightarrow\text{LR expression}
\rightarrow\text{coverage filter}\rightarrow\text{neighborhood CCC}.
\]

In [ ]:
spatial = None
if RUN_FULL_PRIMARY:
    if selected_simulations is not None:
        spatial = selected_simulations[0]
        print("Using frozen oracle-passed tissue, repair =", selected_variant)
    elif not AUTO_REPAIR_ORACLE:
        spatial = simulate_final_spatial(
            spec, PURITY, NICHE_SIZE, NOISE, ORACLE_SEEDS[0],
            device=DEVICE
        )
    else:
        print("No oracle-passed simulation; spatial remains None.")

if spatial is not None:
    print("coordinates   ", spatial.coordinates.shape)
    print("expression    ", spatial.expression.shape)
    print("CS            ", spatial.cs.shape)
    print("communication ", spatial.communication.shape,
          f"~ {spatial.communication.nbytes/1024**2:.1f} MiB")


## 5. Spatial truth：Background、niche domains、cell types

Primary setting 中 5 个 niche 各约 100 cells，剩余为 Background。先量化这个 class imbalance。

In [ ]:
if spatial is not None:
    counts = pd.Series(spatial.labels).value_counts().sort_index()
    label_names = ['Background','N1 Risk','N2 Neutral twin','N3 Protective','N4 Neutral','N5 Neutral']
    truth_counts = pd.DataFrame({
        'label': label_names,
        'cells': [int(counts.get(i,0)) for i in range(6)]
    })
    truth_counts['fraction'] = truth_counts['cells'] / truth_counts['cells'].sum()
    display(truth_counts)

    fig, ax = plt.subplots(figsize=(7,6))
    sc=ax.scatter(spatial.coordinates[:,0], spatial.coordinates[:,1], c=spatial.labels, s=12)
    ax.set_title('True spatial labels')
    ax.set_aspect('equal')
    plt.show()

## 6. CCC filtering：真正进入 \(I_S\) 的 feature mask

当前正式规则只有三层：

1. assay measurability；
2. sender-specific ligand coverage ≥ 10% + receiver-specific receptor coverage ≥ 10%；
3. spatial pair opportunity support。

这里不使用 phenotype、truth active edge、\(H_I\) 或 Top-N。

In [ ]:
if spatial is not None:
    filt = filtering(spec, spatial)

    if selected_oracle is not None:
        feature_indices = np.asarray(selected_oracle["common_feature_indices"], dtype=np.int64)
        blocks = {
            "HC": spatial.cs,
            "HI": spatial.communication[:, feature_indices],
        }
        filt["retained_CCC_common_3_tissues"] = len(feature_indices)
        filt["feature_mask_for_model"] = "intersection of three oracle tissues"
    else:
        blocks, feature_indices = make_blocks(spatial)
        filt["feature_mask_for_model"] = "single-tissue final_mask"

    display(pd.DataFrame({
        "metric": list(filt.keys()),
        "value": [str(v) for v in filt.values()]
    }))
    print("\nFinal block shapes:")
    for name, x in blocks.items():
        print(name, x.shape)


### 注意 filtering precision 的解释

这里的 filter 是 **eligibility filter**，不是 active-edge classifier。因此 retained CCC 中有大量 inactive interaction 是正常的；主要看 signal retention 和 feature burden，而不是要求 precision 很高。

## 7. Oracle identifiability：先问数据“有没有信息”，不要先问 NMF

对模型真正看到的 \(C_S\) 和 retained \(I_S\) 直接做 held-out logistic regression：

- N1/N2：期望 `CS≈random`, `IS high`
- N3/N4：期望 `CS high`, `IS≈random`

这是 **evaluation-only oracle**，不进入训练。

In [ ]:
oracle_df = None
if spatial is not None:
    # Local within-tissue diagnostic (useful for plotting only).
    local_rows = []
    for a,b,name in [(1,2,"N1 vs N2"),(3,4,"N3 vs N4")]:
        cs_auc = heldout_auc(blocks["HC"], spatial.labels, spatial.coordinates, a, b)
        is_auc = heldout_auc(blocks["HI"], spatial.labels, spatial.coordinates, a, b)
        local_rows.append({
            "pair": name,
            "CS_AUC": cs_auc,
            "IS_AUC": is_auc,
            "CS_symmetric": max(cs_auc, 1-cs_auc),
            "IS_symmetric": max(is_auc, 1-is_auc),
        })
    oracle_df = pd.DataFrame(local_rows)
    print("Single-tissue spatial holdout diagnostic:")
    display(oracle_df)

    if selected_oracle is not None:
        print("\nPRIMARY observable gate = leave-one-independent-tissue-out:")
        oracle3_df = pd.DataFrame([
            {
                "seed": r["seed"],
                "N12_CS": r["N1_N2"]["CS_AUC"],
                "N12_CS_sym": max(r["N1_N2"]["CS_AUC"],1-r["N1_N2"]["CS_AUC"]),
                "N12_IS": r["N1_N2"]["IS_AUC"],
                "N34_CS": r["N3_N4"]["CS_AUC"],
                "N34_IS": r["N3_N4"]["IS_AUC"],
                "N34_IS_sym": max(r["N3_N4"]["IS_AUC"],1-r["N3_N4"]["IS_AUC"]),
                "BG_AUC": r["background_vs_niche_CS_IS_AUC"],
            }
            for r in selected_oracle["rows"]
        ])
        display(oracle3_df)
        print("oracle_pass =", oracle_pass(selected_oracle["rows"]))


### 6B. Boundary leakage diagnostic

这里直接比较 all-anchor 与 core-anchor 的 N3/N4 IS AUC。若 all-anchor 可分、core-anchor 接近 0.5，说明泄漏来自 boundary/background/opportunity mixture，而不是 N3/N4 intrinsic molecular CCC truth 不一致。


In [ ]:
if selected_oracle is not None and selected_oracle_all_anchor is not None:
    rows = []
    for rc, ra in zip(selected_oracle["rows"], selected_oracle_all_anchor["rows"]):
        rows.append({
            "seed": rc["seed"],
            "N34_IS_all_anchor": ra["N3_N4"]["IS_AUC"],
            "N34_IS_all_sym": max(ra["N3_N4"]["IS_AUC"], 1-ra["N3_N4"]["IS_AUC"]),
            "N34_IS_core": rc["N3_N4"]["IS_AUC"],
            "N34_IS_core_sym": max(rc["N3_N4"]["IS_AUC"], 1-rc["N3_N4"]["IS_AUC"]),
        })
    boundary_diag = pd.DataFrame(rows)
    display(boundary_diag)


## 8. 再看 centroid：generator truth 和 observable truth 要分开

特别是 composition，cell-level target \(H_C^{truth}\) 经过 neighborhood aggregation 后会变成 observed centroid。两者不能默认相同。

In [ ]:
if spatial is not None:
    cs_centroids=np.stack([blocks['HC'][spatial.labels==k].mean(0) for k in range(6)])
    is_centroids=np.stack([blocks['HI'][spatial.labels==k].mean(0) for k in range(6)])
    generator_hc=np.vstack((np.full(8,.125), spatial.hc_truth))

    hc_shift=np.linalg.norm(cs_centroids-generator_hc,axis=1)
    display(pd.DataFrame({'label':['BG','N1','N2','N3','N4','N5'],
                          '||observed CS centroid - generator HC||':hc_shift}))

    print('Observed CS cosine N1/N2 =',
          np.dot(cs_centroids[1],cs_centroids[2])/(np.linalg.norm(cs_centroids[1])*np.linalg.norm(cs_centroids[2])))
    print('Observed CS cosine N3/N4 =',
          np.dot(cs_centroids[3],cs_centroids[4])/(np.linalg.norm(cs_centroids[3])*np.linalg.norm(cs_centroids[4])))

## 9. ST factorization：C-only / I-only / C+I

当前模型是 shared-\(W\) nonnegative factorization：

\[
C_S\approx W_SH_C,\qquad I_S\approx W_SH_I,\qquad O_S\approx W_SH_O.
\]

`fit_balanced()` 对每个 block 先做 RMS scaling，loss 使用 relative reconstruction error，并在初始化时一次性按对 \(W\) 的 gradient norm 设定固定权重。

In [ ]:
fits={}
reports={}
if spatial is not None:
    method_blocks={
        'C-only': {'HC':blocks['HC']},
        'I-only': {'HI':blocks['HI']},
        'C+I': blocks,
    }
    ws_truth=true_activity(spatial)
    for method, xb in method_blocks.items():
        t0=time.time()
        fit=fit_balanced(xb, seed=MODEL_SEED, device=DEVICE)
        report=evaluate(fit, spatial, ws_truth, blocks)
        fits[method]=fit
        reports[method]=report
        print(method, f'{time.time()-t0:.1f}s',
              'overall_WS=',round(report['overall_ws'],3),
              'sens=',[round(x['sensitivity'],3) for x in report['per_factor']])

## 10. 检查是不是“factor matching”把正确结果配错了

源码 `evaluate()` 用 dictionary similarity 做 Hungarian matching。为了排除 evaluation artifact，下面再用真实 neighborhood activity \(W_S^{truth}\) **只做评价对齐**：

\[
\max_{\pi}\sum_k corr(W_{:,\pi(k)},W^{truth}_{:,k}).
\]

这不参与训练，只回答：“模型是否其实已经学出了正确 spatial factor，只是 label 对错了？”

In [ ]:
def activity_oracle_alignment(fit, ws_truth, labels):
    W=fit.W.numpy()
    corr=np.zeros((W.shape[1],ws_truth.shape[1]))
    for i in range(W.shape[1]):
        for j in range(ws_truth.shape[1]):
            corr[i,j]=_correlation(W[:,i],ws_truth[:,j])
    rows,cols=linear_sum_assignment(-corr)
    order=np.empty(ws_truth.shape[1],dtype=int)
    order[cols]=rows
    aligned=W[:,order]
    aligned=aligned/np.maximum(aligned.sum(1,keepdims=True),1e-12)
    pred=aligned.argmax(1)
    sens=[float(np.mean(pred[labels==k]==k)) for k in range(ws_truth.shape[1])]
    ws=[float(corr[order[k],k]) for k in range(ws_truth.shape[1])]
    return {'order':order,'sensitivity':sens,'WS':ws,'overall_WS':float(np.mean(ws)),'activity':aligned}

oracle_alignment={}
if spatial is not None:
    rows=[]
    for method,fit in fits.items():
        z=activity_oracle_alignment(fit,ws_truth,spatial.labels)
        oracle_alignment[method]=z
        rows.append({'Method':method,'Overall_WS_activity_match':z['overall_WS'],
                     **{f'N{k}_Sens':z['sensitivity'][k] for k in range(6)}})
    display(pd.DataFrame(rows))

### 这里是一个关键分叉

如果：

- oracle classifier AUC 很高；
- activity-oracle matching 后 NMF 仍然恢复很差；

那么失败就不是 simulation 没信息，也不是 factor label 配错，而是：

\[
\boxed{\text{global reconstruction-NMF 没有把这个可分方向分配成独立 niche factor}}
\]

这比继续调 LR signal / purity 更值得优先解决。

## 11. View balancing audit

这里只审计 ST 训练得到的 view scaling / gradient balance。**不再在这里运行 bulk projection。**
上一版把 `infer_activity_balanced()` 和 bulk projection 误放到了这一节，导致 `bulk_expr` 尚未在第 14 节生成就被调用。


In [ ]:
if spatial is not None and "C+I" in fits:
    audit = fits["C+I"].audit
    names = list(audit["alpha"].keys())

    audit_df = pd.DataFrame({
        "view": names,
        "RMS": [audit["RMS"][v] for v in names],
        "alpha": [audit["alpha"][v] for v in names],
        "initial_grad": [audit["initial_gradient_norm"][v] for v in names],
        "weighted_initial_grad": [audit["weighted_initial_gradient_norm"][v] for v in names],
        "final_grad": [audit["final_gradient_norm"][v] for v in names],
        "relative_reconstruction_loss": [audit["loss"][v] for v in names],
    })
    display(audit_df)

    base = audit["weighted_initial_gradient_norm"].get("HI", 1.0)
    final_base = audit["final_gradient_norm"].get("HI", 1.0)
    print(
        "weighted initial gradient ratio vs HI =",
        {v: round(audit["weighted_initial_gradient_norm"][v] / max(base, 1e-12), 3) for v in names}
    )
    print(
        "final gradient ratio vs HI =",
        {v: round(audit["final_gradient_norm"][v] / max(final_base, 1e-12), 3) for v in names}
    )


## 12. True map vs predicted maps

不要只看一个 scalar。空间图最容易暴露 Background 被吞、factor 碎片化、domain collision 等问题。

In [ ]:
if spatial is not None:
    names=['Truth','C-only','I-only','C+I']
    maps=[spatial.labels]
    for method in names[1:]:
        maps.append(np.asarray(reports[method]['predicted']))
    fig,axes=plt.subplots(1,4,figsize=(20,5))
    for ax,name,lab in zip(axes,names,maps):
        ax.scatter(spatial.coordinates[:,0],spatial.coordinates[:,1],c=lab,s=9)
        ax.set_title(name); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    plt.show()

## 13. 自动给出当前 ST 层诊断

这段不是“判论文成败”，只是帮助我们快速确定失败属于哪一层。

In [ ]:
def diagnose_st(oracle_df, reports, oracle_alignment):
    messages=[]
    if oracle_df is None:
        return messages
    x=oracle_df.set_index('pair')
    if x.loc['N3 vs N4','CS_symmetric']>=.90:
        comp_sens=np.mean([reports['C-only']['per_factor'][k]['sensitivity'] for k in (3,4)])
        if comp_sens<.8:
            messages.append('N3/N4 的 observed CS 已高度可分，但 composition-only NMF 未恢复：主要不是 composition signal 缺失。')
    if x.loc['N1 vs N2','IS_symmetric']>=.90:
        ccc_sens=np.mean([reports['I-only']['per_factor'][k]['sensitivity'] for k in (1,2)])
        if ccc_sens<.8:
            messages.append('N1/N2 的 observed IS 已高度可分，但 I-only NMF 未稳定恢复：需要检查 factorization identifiability/optimization。')
    for method,z in oracle_alignment.items():
        original=reports[method]['overall_ws']
        if z['overall_WS']-original>.15:
            messages.append(f'{method}: activity-oracle alignment 明显改善 WS，现有 dictionary matching 可能显著影响评价。')
    if not messages:
        messages.append('当前快速规则没有发现单一明显故障；结合空间图和 per-factor 指标继续判断。')
    return messages

if spatial is not None:
    msgs=diagnose_st(oracle_df,reports,oracle_alignment)
    for i,m in enumerate(msgs,1): print(f'{i}. {m}')

## 13B. 冻结 simulation 后再做 3-seed ST 复现

只有 observable oracle 通过后，这一步才有解释价值。所有 seed 使用相同的 cross-tissue common CCC feature axis。


In [ ]:
three_seed_st = None
if RUN_THREE_SEED_ST and selected_simulations is not None and selected_oracle is not None:
    common_features = np.asarray(selected_oracle["common_feature_indices"], dtype=np.int64)
    st_rows = []

    for rep, sim in enumerate(selected_simulations):
        xb = {
            "HC": sim.cs,
            "HI": sim.communication[:, common_features],
        }
        ws_true = true_activity(sim)

        for method, chosen in {
            "C-only": ("HC",),
            "I-only": ("HI",),
            "C+I": ("HC","HI"),
        }.items():
            fit = fit_balanced({k: xb[k] for k in chosen}, seed=31+rep, device=DEVICE)
            report = evaluate(fit, sim, ws_true, xb)
            st_rows.append({
                "seed": ORACLE_SEEDS[rep],
                "method": method,
                "Overall_WS": report["overall_ws"],
                "BG": report["per_factor"][0]["sensitivity"],
                "N1": report["per_factor"][1]["sensitivity"],
                "N2": report["per_factor"][2]["sensitivity"],
                "N3": report["per_factor"][3]["sensitivity"],
                "N4": report["per_factor"][4]["sensitivity"],
                "N5": report["per_factor"][5]["sensitivity"],
                "collision12": report["collision_12"],
                "collision34": report["collision_34"],
            })

    three_seed_st = pd.DataFrame(st_rows)
    display(three_seed_st)
    print("\nMean by method:")
    display(three_seed_st.groupby("method")[["Overall_WS","BG","N1","N2","N3","N4","N5"]].mean())


## 14. Bulk cohort：先生成 true niche exposure，再生成 bulk observable

患者真实 exposure：

\[
\pi_p\in\Delta^5,
\]

包括 Background + 5 niches。

Bulk 没有空间坐标。`CB` 是 composition，`IB` 是 cell-type-specific ligand/receptor expression 构造出的 **directional molecular communication potential**，不是 spatially realized CCC。

In [ ]:
if spatial is not None:
    pi_true, CB, bulk_expr, surv_time, event, beta_true, eta_true = simulate_final_bulk(
        spec, spatial.hc_truth, NOISE, DATA_SEED+7000, patients=PATIENTS
    )
    print('pi_true   ',pi_true.shape,'row sum err',np.abs(pi_true.sum(1)-1).max())
    print('CB        ',CB.shape)
    print('bulk expr ',bulk_expr.shape)
    print('beta true ',beta_true)
    print('event frac',event.mean())
    print('true C    ',concordance_index(surv_time,event,eta_true))

## 15. Cox sanity：先用 true exposure 做 ALR-Cox

上一版在这里报 `training activity must be nonnegative`，不是 ALR 理论错误，而是复用了一个只接受非负 activity 的通用验证器。

ALR 本来就是 signed covariate，因此这里使用独立的 signed-design Cox fitter。


In [ ]:
def fit_signed_cox(design, time, event, ridge=0.01, iterations=300):
    x = torch.as_tensor(np.asarray(design), dtype=torch.float64)
    t = torch.as_tensor(np.asarray(time), dtype=torch.float64)
    e = torch.as_tensor(np.asarray(event), dtype=torch.float64)

    if x.ndim != 2 or not torch.isfinite(x).all():
        raise ValueError("Cox design must be a finite 2D matrix")
    if t.ndim != 1 or e.ndim != 1 or len(t) != len(x) or len(e) != len(x):
        raise ValueError("time/event must align with design rows")
    if not torch.isfinite(t).all() or not torch.isfinite(e).all() or not bool(e.any()):
        raise ValueError("invalid survival vectors")

    gamma = torch.zeros(x.shape[1], dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS(
        [gamma], max_iter=iterations,
        tolerance_grad=1e-9, tolerance_change=1e-12,
        line_search_fn="strong_wolfe"
    )

    def closure():
        optimizer.zero_grad()
        loss = cox_breslow_loss(x @ gamma, t, e) + ridge * gamma.square().sum()
        if not torch.isfinite(loss):
            raise FloatingPointError("nonfinite Cox objective")
        loss.backward()
        return loss

    optimizer.step(closure)
    return gamma.detach().cpu().numpy().astype(np.float64)


def fit_cox_and_score(design, time, event):
    split = split_patients(len(time), seed=90210)
    tr = split.train
    gamma = fit_signed_cox(design[tr], time[tr], event[tr], ridge=.01)
    out = {"gamma": gamma}
    for part in ("validation","test"):
        idx = getattr(split, part)
        out[part+"_C"] = concordance_index(time[idx], event[idx], design[idx] @ gamma)
    return out


if spatial is not None:
    true_cox = fit_cox_and_score(alr(pi_true), surv_time, event)
    print("TRUE beta             =", TRUE_BETA)
    print("TRUE exposure ALR gamma =", np.round(true_cox["gamma"], 3))
    print("Val C / Test C =",
          round(true_cox["validation_C"],3),
          round(true_cox["test_C"],3))


## 16. ST → bulk projection

这里继续使用 ST 学到的 \(H_C,H_I\)，固定 dictionary，只推断患者 \(W_B\)。

即使 ST gate 没过，本 Notebook 仍允许你**诊断性地**往下跑，但结果不能当成“全链路通过”的证据。

In [ ]:
def infer_activity_balanced(new_blocks, fit, training_blocks, names=("HC","HI"), steps=200):
    """
    Infer nonnegative activity for new samples while mirroring the ST
    C+I objective:
      - use training-view RMS scaling
      - use the frozen alpha weights learned at ST initialization
      - use training scaled block energy for relative reconstruction weighting
    """
    use = tuple(name for name in names if name in new_blocks and name in fit.dictionaries)
    if not use:
        raise ValueError("no common blocks for inference")

    gram = None
    target = None

    for name in use:
        rms = float(fit.audit["RMS"][name])
        alpha = float(fit.audit["alpha"][name])

        x_train_scaled = torch.as_tensor(
            np.asarray(training_blocks[name], dtype=np.float64) / rms,
            dtype=torch.float64,
        )
        x_new_scaled = torch.as_tensor(
            np.asarray(new_blocks[name], dtype=np.float64) / rms,
            dtype=torch.float64,
        )
        h_scaled = torch.as_tensor(
            fit.dictionaries[name].numpy().astype(np.float64) / rms,
            dtype=torch.float64,
        )

        energy = x_train_scaled.square().sum().clamp_min(1e-12)
        weight = alpha / energy

        g = weight * (h_scaled @ h_scaled.T)
        t = weight * (x_new_scaled @ h_scaled.T)
        gram = g if gram is None else gram + g
        target = t if target is None else target + t

    lipschitz = gram.abs().sum(1).max().clamp_min(torch.finfo(torch.float64).tiny)
    step = 1.0 / lipschitz

    activity = torch.zeros_like(target)
    extrapolated = activity.clone()
    momentum = 1.0

    for _ in range(steps):
        updated = torch.relu(extrapolated - step * (extrapolated @ gram - target))
        next_momentum = (1.0 + np.sqrt(1.0 + 4.0 * momentum**2)) / 2.0
        extrapolated = updated + ((momentum - 1.0) / next_momentum) * (updated - activity)
        activity = updated
        momentum = next_momentum

    return activity.cpu().numpy()


WB = None
IB = None

if spatial is not None and "C+I" in fits:
    required = ["bulk_expr", "CB", "pi_true", "surv_time", "event"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Bulk cohort has not been generated yet. "
            "Run Section 14 before Section 16. Missing: " + ", ".join(missing)
        )

    full_fit = fits["C+I"]
    order = np.asarray(reports["C+I"]["order"])

    bulk_all = bulk_potential(bulk_expr, spec)
    IB = bulk_all[:, feature_indices]
    print("IB retained shape =", IB.shape)

    raw_wb = infer_activity_balanced(
        {"HC": CB, "HI": IB},
        full_fit,
        blocks,
        names=("HC", "HI"),
    )

    WB = raw_wb[:, order]
    WB = WB / np.maximum(WB.sum(1, keepdims=True), 1e-12)

    wb_corr = [_correlation(WB[:, k], pi_true[:, k]) for k in range(6)]
    display(pd.DataFrame({
        "niche": ["BG", "Risk N1", "Neutral N2", "Protective N3", "Neutral N4", "Neutral N5"],
        "WB correlation": wb_corr,
    }))


## 17. Raw-W vs ALR Cox

Raw compositional \(W_B\) 加 intercept 天然存在 rank deficiency；正式解释应使用 Background-reference ALR。

In [ ]:
if spatial is not None and WB is not None:
    raw_stats = collinearity(WB)
    alr_stats = collinearity(alr(WB))
    print("Raw-W condition number =", raw_stats["condition_number_with_intercept"])
    print("ALR condition number   =", alr_stats["condition_number_with_intercept"])

    inferred_cox = fit_cox_and_score(alr(WB), surv_time, event)
    print("INFERRED WB ALR gamma =", np.round(inferred_cox["gamma"], 3))
    print(
        "Val C / Test C =",
        round(inferred_cox["validation_C"], 3),
        round(inferred_cox["test_C"], 3),
    )
else:
    print("WB not available; Section 17 skipped.")


## 18. 最终层级化总结

必须把 4 层分开，不允许“C-index 不好”就回头说 CCC simulation 错，也不允许 ST 失败却用 bulk/Cox 的偶然结果证明方法成立。

In [ ]:
if spatial is not None:
    level1 = (
        oracle_df.set_index('pair').loc['N1 vs N2','CS_symmetric'] <= .60 and
        oracle_df.set_index('pair').loc['N1 vs N2','IS_symmetric'] >= .90 and
        oracle_df.set_index('pair').loc['N3 vs N4','CS_symmetric'] >= .90 and
        oracle_df.set_index('pair').loc['N3 vs N4','IS_symmetric'] <= .60
    )
    full=reports['C+I']
    level2=(full['overall_ws']>=.75 and
            all(full['per_factor'][k]['sensitivity']>=.8 for k in range(5)) and
            full['collision_12'] is False and full['collision_34'] is False)
    level3=('WB' in globals() and np.mean(wb_corr)>=.5 and wb_corr[1]>=.5 and wb_corr[3]>=.5)
    level4=('inferred_cox' in globals() and inferred_cox['gamma'][0]>0 and inferred_cox['gamma'][2]<0)
    display(pd.DataFrame({
        'Level':['1 Observable identifiability','2 ST niche discovery','3 ST→bulk transfer','4 ALR phenotype'],
        'pass':[level1,level2,level3,level4]
    }))

## 19. 三组织 oracle 已前置为硬 gate

v2 不再把 3-tissue oracle 放到最后作为可选诊断；它已经在任何 ST fitting 之前运行。


In [ ]:
if selected_oracle is not None:
    print('selected repair =', selected_variant)
    print('common retained CCC =', selected_oracle['common_feature_count'])
    print('oracle_pass =', oracle_pass(selected_oracle['rows']))


## 20. Repair history：这里只用于理解 generator，不用于调模型

`oracle_attempts` 保存每一步 observable-only repair 的结果。选择过程完全发生在 ST/NMF、bulk 和 phenotype 之前。


In [ ]:
if oracle_attempts:
    display(pd.DataFrame([{
        'variant': x['variant'],
        'passed': x['passed'],
        'common_features': x['common_features'],
    } for x in oracle_attempts]))


## 21. 推荐运行顺序

直接 **Restart Kernel → Run All**。

先看 `3B` 的 **CORE-neighborhood oracle**。`All-anchor diagnostic` 只用于观察 boundary leakage，不再决定 generator 是否通过。

如果 core oracle 通过，再看三 seed ST。训练仍使用全部 anchors，包括 boundary；我们只是避免把混合 neighborhood 当成纯 niche 去审计 simulation truth。

之后再依次看 ST→bulk 和 ALR-Cox。
